In [ ]:
# Install required libraries
!pip install -q efficientnet_pytorch albumentations

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from efficientnet_pytorch import EfficientNet
from albumentations import Compose, Rotate, HorizontalFlip, VerticalFlip, Normalize
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Configuration
DATA_DIR = '/content/drive/MyDrive/aptos2019/'  # path
CSV_PATH = os.path.join(DATA_DIR, 'train.csv')  # CSV path
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 25
NUM_FOLDS = 5
SEED = 42

In [ ]:
class AptosDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.df = df
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['id_code']
        img_path = os.path.join(DATA_DIR, 'train_images', f'{img_name}.png')

        # Read and preprocess image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Circle crop
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        circles = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, 1, 200,
                                 param1=30, param2=10, minRadius=0, maxRadius=0)
        if circles is not None:
            x, y, r = np.uint16(np.around(circles[0][0]))
            image = image[y-r:y+r, x-r:x+r]
        else:  # Fallback to center crop
            h, w = image.shape[:2]
            min_dim = min(h, w)
            start_h = (h - min_dim) // 2
            start_w = (w - min_dim) // 2
            image = image[start_h:start_h+min_dim, start_w:start_w+min_dim]

        image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        if self.is_test:
            return image
        else:
            label = self.df.iloc[idx]['diagnosis']
            return image, torch.tensor(label, dtype=torch.long)

In [ ]:
# Define transforms
train_transform = Compose([
    Rotate(limit=360, p=0.5),
    HorizontalFlip(p=0.5),
    VerticalFlip(p=0.5),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transform = Compose([
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

In [ ]:
# Load and prepare data
df = pd.read_csv(CSV_PATH)

# Undersample class 0
class_0 = df[df['diagnosis'] == 0]
sampled_class_0 = class_0.sample(frac=0.5, random_state=SEED)
df = pd.concat([df[df['diagnosis'] != 0], sampled_class_0])

# Prepare k-fold
skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(df, df['diagnosis']))

In [ ]:
def train_model(fold):
    train_idx, val_idx = folds[fold]
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    train_dataset = AptosDataset(train_df, transform=train_transform)
    val_dataset = AptosDataset(val_df, transform=val_transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = EfficientNet.from_pretrained('efficientnet-b5')
    model._fc = nn.Linear(model._fc.in_features, 5)
    model = model.cuda()

    # Warm-up classifier head
    for param in model.parameters():
        param.requires_grad = False
    for param in model._fc.parameters():
        param.requires_grad = True

    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    # Warm-up phase
    for epoch in range(5):
        model.train()
        for inputs, labels in tqdm(train_loader):
            inputs = inputs.cuda()
            labels = labels.cuda()

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    # Train full model
    for param in model.parameters():
        param.requires_grad = True

    best_score = 0
    for epoch in range(20):
        model.train()
        train_loss = 0
        for inputs, labels in tqdm(train_loader):
            inputs = inputs.cuda()
            labels = labels.cuda()

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.cuda()
                labels = labels.cuda()

                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = correct / total
        if val_acc > best_score:
            best_score = val_acc
            torch.save(model.state_dict(), f'best_model_fold{fold}.pth')

    return best_score

In [ ]:
# Train all folds
for fold in range(NUM_FOLDS):
    print(f'Training fold {fold}')
    train_model(fold)

In [ ]:
def tta_predict(image, model, n_tta=10):
    tta_transforms = Compose([
        Rotate(limit=360, p=1),
        HorizontalFlip(p=0.5),
        VerticalFlip(p=0.5),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])

    model.eval()
    preds = []
    with torch.no_grad():
        for _ in range(n_tta):
            augmented = tta_transforms(image=image)
            img = augmented['image'].unsqueeze(0).cuda()
            output = model(img)
            preds.append(torch.softmax(output, dim=1).cpu().numpy())
    return np.mean(preds, axis=0)

In [ ]:
# Generate submission
test_dir = os.path.join(DATA_DIR, 'test_images')
test_files = os.listdir(test_dir)
submission = pd.DataFrame(columns=['id_code', 'diagnosis'])

for file in test_files:
    img_path = os.path.join(test_dir, file)
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Preprocess (same as dataset class)
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    circles = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, 1, 200,
                         param1=30, param2=10, minRadius=0, maxRadius=0)

    if circles is not None:
       x, y, r = np.uint16(np.around(circles[0][0]))
      image = image[y-r:y+r, x-r:x+r]
    else:  # Fallback to center crop
        h, w = image.shape[:2]
        min_dim = min(h, w)
        start_h = (h - min_dim) // 2
        start_w = (w - min_dim) // 2
        image = image[start_h:start_h+min_dim, start_w:start_w+min_dim]

    image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))

    final_pred = np.zeros(5)
    for fold in range(NUM_FOLDS):
        model = EfficientNet.from_pretrained('efficientnet-b5')
        model._fc = nn.Linear(model._fc.in_features, 5)
        model.load_state_dict(torch.load(f'best_model_fold{fold}.pth'))
        model = model.cuda()

        pred = tta_predict(image, model)
        final_pred += pred[0]

    final_pred /= NUM_FOLDS
    submission = submission.append({
        'id_code': file.split('.')[0],
        'diagnosis': np.argmax(final_pred)
    }, ignore_index=True)

submission.to_csv('submission.csv', index=False)